# Python Generators and Iterators — Interview Preparation

Covers: iterator protocol, generator functions, yield from, generator expressions, send/throw/close, itertools, and interview Q&A.

## 1. Iterator Protocol

In [ ]:
# Custom iterator implementing __iter__ and __next__
class CountDown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self  # iterator returns itself

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


cd = CountDown(5)
print('for loop:', list(cd))

# Manual iteration
cd2 = CountDown(3)
print('manual:', next(cd2), next(cd2), next(cd2))

# next() with default — no StopIteration
print('next with default:', next(cd2, 'done'))  # 'done'

# Iterable vs Iterator
lst = [1, 2, 3]
it1 = iter(lst)
it2 = iter(lst)
print('it1 is it2:', it1 is it2)  # False — fresh iterators
print('iter(it1) is it1:', iter(it1) is it1)  # True — iterator returns self

## 2. Generator Functions

In [ ]:
# Generator function — uses yield
def fibonacci():
    """Infinite Fibonacci generator"""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


import itertools
fib = fibonacci()
first_10 = list(itertools.islice(fib, 10))
print('First 10 Fibonacci:', first_10)

# Generator is lazy — no computation until next()
def squares(n):
    print('Generator started')
    for i in range(n):
        print(f'  yielding {i**2}')
        yield i ** 2

gen = squares(3)  # nothing printed yet!
print('Created generator')
print('First:', next(gen))
print('Second:', next(gen))

In [ ]:
# yield from — delegate to sub-generator
def flatten(nested):
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)  # recursive delegation
        else:
            yield item


data = [1, [2, [3, 4]], [5, 6], 7]
print('Flattened:', list(flatten(data)))

# yield from with chain
def chain(*iterables):
    for it in iterables:
        yield from it

print('Chained:', list(chain([1, 2], [3, 4], [5])))

## 3. Generator Expressions — Memory Efficiency

In [ ]:
import sys

# Memory comparison
lst = [x**2 for x in range(10000)]
gen = (x**2 for x in range(10000))

print(f'List size: {sys.getsizeof(lst):,} bytes')
print(f'Generator size: {sys.getsizeof(gen)} bytes')

# Use with aggregation functions — no intermediate list
total = sum(x**2 for x in range(10000))
maximum = max(x**2 for x in range(10000))
print(f'sum: {total}, max: {maximum}')

# Generator can only be iterated ONCE
gen = (x for x in range(3))
print('First pass:', list(gen))   # [0, 1, 2]
print('Second pass:', list(gen))  # []  — exhausted!

## 4. send(), throw(), close()

In [ ]:
# send() — send a value INTO the generator
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        value = yield average  # yield sends average out, receives value in
        if value is None:
            break
        total += value
        count += 1
        average = total / count


avg = running_average()
next(avg)           # prime the generator (advance to first yield)
print('After 10:', avg.send(10))   # 10.0
print('After 20:', avg.send(20))   # 15.0
print('After 30:', avg.send(30))   # 20.0

In [ ]:
# close() — throw GeneratorExit
def resource_generator():
    print('Resource acquired')
    try:
        while True:
            yield 'working'
    except GeneratorExit:
        print('Resource released (cleanup)')  # runs on close()


gen = resource_generator()
next(gen)
next(gen)
gen.close()  # triggers cleanup

## 5. itertools

In [ ]:
import itertools

# chain — concatenate iterables
print('chain:', list(itertools.chain([1, 2], [3, 4], [5])))

# islice — lazy slicing
print('islice(count, 5):', list(itertools.islice(itertools.count(0), 5)))

# product — Cartesian product
print('product:', list(itertools.product([1, 2], ['a', 'b'])))

# combinations — order doesn't matter, no repetition
print('combinations([1,2,3], 2):', list(itertools.combinations([1, 2, 3], 2)))

# permutations — order matters
print('permutations([1,2,3], 2):', list(itertools.permutations([1, 2, 3], 2)))

# accumulate — running totals
import operator
print('accumulate (sum):', list(itertools.accumulate([1, 2, 3, 4, 5])))
print('accumulate (product):', list(itertools.accumulate([1, 2, 3, 4, 5], operator.mul)))

In [ ]:
# groupby — group consecutive elements (MUST be sorted first!)
data = [('a', 1), ('a', 2), ('b', 3), ('b', 4), ('a', 5)]

print('groupby (unsorted — wrong!):')
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(f'  {key}: {list(group)}')

print('groupby (sorted — correct!):')
sorted_data = sorted(data, key=lambda x: x[0])
for key, group in itertools.groupby(sorted_data, key=lambda x: x[0]):
    print(f'  {key}: {list(group)}')

## 6. Interview Practice

In [ ]:
# Q: Implement a chunking generator
def chunks(iterable, size):
    """Split iterable into chunks of given size"""
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, size))
        if not chunk:
            break
        yield chunk


print('chunks:', list(chunks(range(10), 3)))

# Q: Lazy pipeline
def read_numbers():
    yield from range(100)

def filter_even(nums):
    return (n for n in nums if n % 2 == 0)

def square(nums):
    return (n**2 for n in nums)

# Compose pipeline — nothing computed until consumed
pipeline = square(filter_even(read_numbers()))
print('First 5 even squares:', list(itertools.islice(pipeline, 5)))